# Лабораторная работа: Многослойный перцептрон (MLP)

**Задача:** бинарная классификация опухолей молочной железы (Malignant / Benign) на основе числовых признаков.

**Датасет:** `breast-cancer.csv` — 569 наблюдений, 30 числовых признаков.

**Фреймворк:** PyTorch

## Признаки, по которым обучается модель

Модель получает **30 числовых признаков**. Они образованы из 10 характеристик клеточных ядер:

| Признак | Что означает |
|---|---|
| `radius` | Радиус ядра — расстояние от центра до границы |
| `texture` | Текстура — разброс оттенков серого внутри ядра |
| `perimeter` | Длина границы ядра |
| `area` | Площадь ядра |
| `smoothness` | Гладкость границы ядра |
| `compactness` | Компактность формы ядра |
| `concavity` | Выраженность углублений на контуре |
| `concave points` | Количество вогнутых участков контура |
| `symmetry` | Симметричность формы ядра |
| `fractal_dimension` | Сложность и неровность контура |

Каждый признак представлен в трёх вариантах:

| Суффикс | Значение | Пример |
|---|---|---|
| `_mean` | Среднее значение | `radius_mean` — средний радиус |
| `_se` | Стандартная ошибка, то есть мера разброса | `radius_se` — разброс радиуса |
| `_worst` | Среднее трёх наибольших значений | `radius_worst` — наиболее выраженный радиус |

> `diagnosis` — правильный ответ для обучения: `B` — доброкачественная опухоль, `M` — злокачественная.


---
## 0. Импорт библиотек


In [64]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    roc_auc_score, roc_curve
)
from typing import List





Константы

In [65]:
DATA_PATH = 'breast-cancer.csv'
BATCH_SIZE = 32
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

---
## 1. Загрузка и первичный анализ данных


In [66]:


df = pd.read_csv(DATA_PATH)
print(f'Size: {df.shape}')
df.head()

Size: (569, 32)


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [67]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 32 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             5

In [68]:
# транспонирования  таблицы описательной статистики.
df.describe().T

,count,mean,std,min,25%,50%,75%,max
id,569.0,3.037183e+07,1.250206e+08,8670.000000,869218.000000,906024.000000,8.813129e+06,9.113205e+08
radius_mean,569.0,1.412729e+01,3.524049e+00,6.981000,11.700000,13.370000,1.578000e+01,2.811000e+01
texture_mean,569.0,1.928965e+01,4.301036e+00,9.710000,16.170000,18.840000,2.180000e+01,3.928000e+01
perimeter_mean,569.0,9.196903e+01,2.429898e+01,43.790000,75.170000,86.240000,1.041000e+02,1.885000e+02
area_mean,569.0,6.548891e+02,3.519141e+02,143.500000,420.300000,551.100000,7.827000e+02,2.501000e+03
smoothness_mean,569.0,9.636028e-02,1.406413e-02,0.052630,0.086370,0.095870,1.053000e-01,1.634000e-01
compactness_mean,569.0,1.043410e-01,5.281276e-02,0.019380,0.064920,0.092630,1.304000e-01,3.454000e-01
concavity_mean,569.0,8.879932e-02,7.971981e-02,0.000000,0.029560,0.061540,1.307000e-01,4.268000e-01
concave points_mean,569.0,4.891915e-02,3.880284e-02,0.000000,0.020310,0.033500,7.400000e-02,2.012000e-01
symmetry_mean,569.0,1.811619e-01,2.741428e-02,0.106000,0.161900,0.179200,1.957000e-01,3.040000e-01


In [69]:
print('Пропущенные значения:')
print(df.isnull().sum()[df.isnull().sum() > 0])

if df.isnull().sum().sum() == 0:
    print('Пропуски отсутствуют')

Пропущенные значения:
Series([], dtype: int64)
Пропуски отсутствуют


In [70]:
print('Распределение целевой переменной:')
print(df['diagnosis'].value_counts())

# абсолютные значения (количество штук) в относительные частоты (доли от 1 или проценты).
print(f'\nДоля классов:')
print(df['diagnosis'].value_counts(normalize=True).round(3))

Распределение целевой переменной:
diagnosis
B    357
M    212
Name: count, dtype: int64

Доля классов:
diagnosis
B    0.627
M    0.373
Name: proportion, dtype: float64


In [71]:
feature_cols = [c for c in df.columns if c not in ('id', 'diagnosis')]
mean_cols = [c for c in feature_cols if c.endswith('_mean')]

---
## 2. Визуализация данных


M — Malignant, то есть злокачественная опухоль.
B — Benign, то есть доброкачественная опухоль.

### Столбчатая диаграмма классов


In [72]:
class_counts = df['diagnosis'].value_counts()
fig = go.Figure(go.Bar(
    x=class_counts.index,
    y=class_counts.values,
    marker_color=['#2ecc71', '#e74c3c'],
    text=class_counts.values,
    textposition='auto'
))
fig.update_layout(
    title='Распределение классов (M — злокачественная, B — доброкачественная)',
    xaxis_title='Диагноз',
    yaxis_title='Количество',
    template='plotly_white',
    width=600, height=400
)
fig.show()

Что показывает: распределения средних признаков отдельно для B и M. Что даёт: чем меньше перекрытие распределений, тем полезнее признак для классификации.


In [73]:


fig = make_subplots(rows=2, cols=5, subplot_titles=mean_cols)
for i, col in enumerate(mean_cols):
    row, col_idx = i // 5 + 1, i % 5 + 1
    for label, color in [('M', '#e74c3c'), ('B', '#2ecc71')]:
        subset = df[df['diagnosis'] == label]
        fig.add_trace(go.Histogram(
            x=subset[col], name=label, marker_color=color,
            opacity=0.6, nbinsx=20, showlegend=(i == 0)
        ), row=row, col=col_idx)
    fig.update_xaxes(title_text=col, row=row, col=col_idx)

fig.update_layout(
    title_text='Гистограммы средних признаков по классам',
    barmode='overlay', template='plotly_white',
    height=600, width=1200
)
fig.show()


### Boxplot (ящик с усами)
Каждый ящик показывает медиану, разброс и выбросы признака для каждого класса. Если ящики M и B не пересекаются — признак хорошо разделяет классы. Точки за пределами усов — это выбросы (нетипичные значения).
```
    |        ← верхний "ус" — максимум (без выбросов)
  ┌─┤
  │ │        ← верхняя граница коробки — 75% значений ниже этой линии
  │─│        ← линия внутри — МЕДИАНА (середина данных)
  │ │        ← нижняя граница коробки — 25% значений ниже этой линии
  └─┤
    |        ← нижний "ус" — минимум (без выбросов)

    ●        ← точки за пределами усов — выбросы (необычные значения)
```

Что показывает: медиану, разброс и выбросы признаков для B и M. Что даёт: помогает сравнить типичные значения классов и найти хорошо разделяющие признаки.


In [74]:
fig = make_subplots(rows=2, cols=5, subplot_titles=mean_cols)
for i, col in enumerate(mean_cols):
    row, col_idx = i // 5 + 1, i % 5 + 1
    for label, color in [('M', '#e74c3c'), ('B', '#2ecc71')]:
        subset = df[df['diagnosis'] == label]
        fig.add_trace(go.Box(
            y=subset[col], name=label, marker_color=color,
            showlegend=(i == 0)
        ), row=row, col=col_idx)

fig.update_layout(
    title_text='Boxplot средних признаков по классам',
    template='plotly_white', height=600, width=1200
)
fig.show()


### Кореляция

In [75]:
corr = df[mean_cols].corr()

fig = go.Figure(go.Heatmap(
    z=corr.values,
    x=mean_cols,
    y=mean_cols,
    colorscale='RdBu',
    zmin=-1,
    zmax=1,
    colorbar=dict(title='Корреляция')
))

for row, y_name in enumerate(mean_cols):
    for col, x_name in enumerate(mean_cols):
        value = corr.iloc[row, col]
        fig.add_annotation(
            x=x_name,
            y=y_name,
            text=f'{value:.2f}',
            showarrow=False,
            font=dict(color='white' if abs(value) > 0.55 else 'black', size=10)
        )

fig.update_layout(
    title='Корреляционная матрица (mean-признаки)',
    template='plotly_white',
    width=900,
    height=800,
    xaxis=dict(tickangle=-45),
    yaxis=dict(autorange='reversed')
)
fig.show()


### PCA
<!-- Что показывает: объекты в пространстве двух главных компонент PCA. Что даёт: позволяет визуально оценить разделимость классов и зоны возможных ошибок. -->


In [76]:
from sklearn.decomposition import PCA

scaler_pca = StandardScaler()
X_pca = scaler_pca.fit_transform(df[feature_cols])
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_pca)

fig = go.Figure()
for label, color, symbol in [('M', '#e74c3c', 'x'), ('B', '#2ecc71', 'circle')]:
    mask = df['diagnosis'] == label
    fig.add_trace(go.Scatter(
        x=X_2d[mask, 0], y=X_2d[mask, 1],
        mode='markers', name=label,
        marker=dict(color=color, symbol=symbol, size=7, opacity=0.7)
    ))

fig.update_layout(
    title='PCA-проекция данных',
    xaxis_title=f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',
    yaxis_title=f'PC2 ({pca.explained_variance_ratio_[1]:.1%})',
    template='plotly_white', width=800, height=600
)
fig.show()


---
## 3. Предобработка и разделение данных

Разбиение: **train / validation / test = 60% / 20% / 20%**

- **train** — для обучения модели (подгонка весов);
- **validation** — для подбора гиперпараметров и ранней остановки (контроль переобучения);
- **test** — финальная, независимая оценка качества модели.


In [77]:
X = df[feature_cols].values
le = LabelEncoder()
y = le.fit_transform(df['diagnosis'])  # B=0, M=1


X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=SEED, stratify=y_train_val
)

print(f'Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}')
print(f'Доля M в train: {y_train.mean():.3f}')
print(f'Доля M в val:   {y_val.mean():.3f}')
print(f'Доля M в test:  {y_test.mean():.3f}')

Train: 341, Val: 114, Test: 114
Доля M в train: 0.372
Доля M в val:   0.377
Доля M в test:  0.368


In [78]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

def to_tensors(X, y):
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32).unsqueeze(1)

X_train_t, y_train_t = to_tensors(X_train, y_train)
X_val_t, y_val_t = to_tensors(X_val, y_val)
X_test_t, y_test_t = to_tensors(X_test, y_test)



train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH_SIZE)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=BATCH_SIZE)


In [79]:


class MLP(nn.Module):

    def __init__(self, inputDim: int,
                 hiddenShape: List[int],
                 outputDim: int = 1,
                 fActivation = nn.ReLU,
                 dropoutRate: float = 0.3):

        super().__init__()

        # вход: 30 → первый скрытый размер
        self.inputModule = nn.Sequential(
            nn.Linear(inputDim, hiddenShape[0]),
            nn.BatchNorm1d(hiddenShape[0]),
            fActivation(),
            nn.Dropout(dropoutRate),
        )

        # скрытые: переходы между соседними размерами из списка
        self.hiddenModule = nn.Sequential(
            *[nn.Sequential(
                nn.Linear(hiddenShape[i], hiddenShape[i + 1]),
                nn.BatchNorm1d(hiddenShape[i + 1]),
                fActivation(),
                nn.Dropout(dropoutRate),
              ) for i in range(len(hiddenShape) - 1)]
        )

        # выход: последний скрытый размер → 1 число (логит)
        self.outputModule = nn.Linear(hiddenShape[-1], outputDim)

    def forward(self, x):
        x = self.inputModule(x)
        x = self.hiddenModule(x)
        x = self.outputModule(x)
        return x



INPUT_DIM = X_train.shape[1]
HIDDEN_SIZES = [64, 32]
EPOCHS = 100
LR = 1e-3

model = MLP(INPUT_DIM, HIDDEN_SIZES, outputDim=1, fActivation=nn.ReLU, dropoutRate=0.3)
print(model)


MLP(
  (inputModule): Sequential(
    (0): Linear(in_features=30, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
  )
  (hiddenModule): Sequential(
    (0): Sequential(
      (0): Linear(in_features=64, out_features=32, bias=True)
      (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0.3, inplace=False)
    )
  )
  (outputModule): Linear(in_features=32, out_features=1, bias=True)
)


In [ ]:
def evaluate(model, loader, criterion):
    loss_sum, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        logits = model(xb)
        loss_sum += criterion(logits, yb).item() * xb.size(0)
        preds = (torch.sigmoid(logits) > 0.5).float()
        correct += (preds == yb).sum().item()
        total += yb.size(0)
    return loss_sum / total, correct / total


def train_model(model, train_loader, val_loader, epochs, lr):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(epochs):
        # Обучение
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        # Метрики
        model.eval()
        with torch.no_grad():
            t_loss, t_acc = evaluate(model, train_loader, criterion)
            v_loss, v_acc = evaluate(model, val_loader, criterion)

        history['train_loss'].append(t_loss)
        history['val_loss'].append(v_loss)
        history['train_acc'].append(t_acc)
        history['val_acc'].append(v_acc)

    return history

---
## 4. Построение и обучение MLP

Архитектура: настраиваемый MLP с параметрами `hidden_sizes` (список размеров скрытых слоёв), `dropout_rate`.


In [81]:
history = train_model(model, train_loader, val_loader, EPOCHS, LR)

### 4.1 Визуализация обучения и проверка переобучения

Графики показывают, как менялись ошибка (loss) и точность (accuracy) на обучающей и валидационной выборках по эпохам. Если train loss падает, а val loss начинает расти — модель переобучается (запоминает тренировочные данные вместо обобщения).


Что показывает: loss и accuracy на train и validation по эпохам. Что даёт: показывает сходимость модели и признаки переобучения.


In [82]:
def plot_history(history, title=''):
    epochs = list(range(1, len(history['train_loss']) + 1))

    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=[f'Функция потерь {title}', f'Точность {title}'])

    fig.add_trace(go.Scatter(x=epochs, y=history['train_loss'], name='Train Loss',
                             line=dict(color='#1f77b4')), row=1, col=1)
    fig.add_trace(go.Scatter(x=epochs, y=history['val_loss'], name='Val Loss',
                             line=dict(color='#ff7f0e')), row=1, col=1)

    fig.add_trace(go.Scatter(x=epochs, y=history['train_acc'], name='Train Accuracy',
                             line=dict(color='#1f77b4')), row=1, col=2)
    fig.add_trace(go.Scatter(x=epochs, y=history['val_acc'], name='Val Accuracy',
                             line=dict(color='#ff7f0e')), row=1, col=2)

    fig.update_xaxes(title_text='Эпоха', row=1, col=1)
    fig.update_xaxes(title_text='Эпоха', row=1, col=2)
    fig.update_yaxes(title_text='Loss', row=1, col=1)
    fig.update_yaxes(title_text='Accuracy', row=1, col=2)

    fig.update_layout(height=500, width=1000, template='plotly_white')
    fig.show()

    gap = history['train_acc'][-1] - history['val_acc'][-1]
    print(f'Train Acc: {history["train_acc"][-1]:.4f}')
    print(f'Val Acc:   {history["val_acc"][-1]:.4f}')
    print(f'Разрыв (train - val): {gap:.4f}')
    if gap > 0.05:
        print('⚠ Наблюдается переобучение (разрыв > 5%)')
    else:
        print('✓ Переобучение не выражено')

plot_history(history, '(базовая модель: [64, 32])')

Train Acc: 0.9853
Val Acc:   0.9825
Разрыв (train - val): 0.0029
✓ Переобучение не выражено


### 4.2 Оценка на тестовой выборке


In [83]:
def evaluate_model(model, test_loader):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            logits = model(xb)
            probs = torch.sigmoid(logits).numpy()
            preds = (probs > 0.5).astype(int)
            all_probs.extend(probs.flatten())
            all_preds.extend(preds.flatten())
            all_labels.extend(yb.numpy().flatten())
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

y_true, y_pred, y_prob = evaluate_model(model, test_loader)

print('=== Результаты на тестовой выборке ===')
print(f'Accuracy:  {accuracy_score(y_true, y_pred):.4f}')
print(f'Precision: {precision_score(y_true, y_pred):.4f}')
print(f'Recall:    {recall_score(y_true, y_pred):.4f}')
print(f'F1-score:  {f1_score(y_true, y_pred):.4f}')
print(f'ROC AUC:   {roc_auc_score(y_true, y_prob):.4f}')
print()
print(classification_report(y_true, y_pred, target_names=le.classes_))


=== Результаты на тестовой выборке ===
Accuracy:  0.9649
Precision: 0.9750
Recall:    0.9286
F1-score:  0.9512
ROC AUC:   0.9907

              precision    recall  f1-score   support

           B       0.96      0.99      0.97        72
           M       0.97      0.93      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



Что показывает: матрица ошибок и ROC-кривую. Что даёт: показывает типы ошибок модели и качество разделения классов при разных порогах.


In [84]:
cm = confusion_matrix(y_true, y_pred)
fig = make_subplots(rows=1, cols=2, subplot_titles=['Confusion Matrix', 'ROC-кривая'],
                    specs=[[{'type': 'heatmap'}, {'type': 'scatter'}]])
fig.add_trace(go.Heatmap(z=cm, x=list(le.classes_), y=list(le.classes_),
                         colorscale='Blues', showscale=False, text=cm,
                         texttemplate='%{text}', textfont=dict(size=18)), row=1, col=1)
fig.update_xaxes(title_text='Предсказано', row=1, col=1)
fig.update_yaxes(title_text='Истинное', autorange='reversed', row=1, col=1)
fpr, tpr, _ = roc_curve(y_true, y_prob)
auc_value = roc_auc_score(y_true, y_prob)
fig.add_trace(go.Scatter(x=fpr, y=tpr, name=f'AUC = {auc_value:.3f}'), row=1, col=2)
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], name='Случайный классификатор',
                         line=dict(color='gray', dash='dash')), row=1, col=2)
fig.update_xaxes(title_text='False Positive Rate', row=1, col=2)
fig.update_yaxes(title_text='True Positive Rate', row=1, col=2)
fig.update_layout(template='plotly_white', height=500, width=1100)
fig.show()


---
## 5. Исследование влияния глубины и ширины MLP


In [85]:
architectures = {
    'Shallow-Narrow [16]':          [16],
    'Shallow-Wide [128]':           [128],
    'Medium [64, 32]':              [64, 32],
    'Deep-Narrow [32, 16, 8]':      [32, 16, 8],
    'Deep-Wide [128, 64, 32]':      [128, 64, 32],
    'Very Deep [64, 32, 16, 8, 4]': [64, 32, 16, 8, 4],
}

results = {}

for name, hidden in architectures.items():
    torch.manual_seed(SEED)
    m = MLP(INPUT_DIM, hidden, dropout_rate=0.3)
    h = train_model(m, train_loader, val_loader, EPOCHS, LR)
    y_true_exp, y_pred_exp, y_prob_exp = evaluate_model(m, test_loader)
    acc = accuracy_score(y_true_exp, y_pred_exp)
    f1 = f1_score(y_true_exp, y_pred_exp)
    auc = roc_auc_score(y_true_exp, y_prob_exp)
    n_params = sum(p.numel() for p in m.parameters())
    results[name] = {
        'hidden': hidden,
        'params': n_params,
        'test_acc': acc,
        'test_f1': f1,
        'test_auc': auc,
        'history': h
    }
    print(f'{name:35s} | params={n_params:>6,} | acc={acc:.4f} | f1={f1:.4f} | auc={auc:.4f}')


TypeError: __init__() got an unexpected keyword argument 'dropout_rate'

In [ ]:
res_df = pd.DataFrame({
    k: {'Параметры': v['params'], 'Accuracy': v['test_acc'],
         'F1': v['test_f1'], 'AUC': v['test_auc']}
    for k, v in results.items()
}).T
res_df = res_df.sort_values('Accuracy', ascending=False)
print(res_df.to_string())

                              Параметры  Accuracy        F1       AUC
Shallow-Narrow [16]               545.0  0.982456  0.975610  0.994378
Deep-Narrow [32, 16, 8]          1777.0  0.973684  0.962963  0.997024
Very Deep [64, 32, 16, 8, 4]     5017.0  0.973684  0.963855  0.996362
Shallow-Wide [128]               4353.0  0.964912  0.951220  0.992063
Medium [64, 32]                  4289.0  0.964912  0.951220  0.990741
Deep-Wide [128, 64, 32]         14785.0  0.956140  0.939759  0.992725


Что показывает: train loss и validation loss для каждой архитектуры. Что даёт: позволяет сравнить стабильность обучения и склонность моделей к переобучению.


In [ ]:
names_list = list(results.keys())
fig = make_subplots(rows=2, cols=3, subplot_titles=[
    f'{name}<br>acc={results[name]["test_acc"]:.3f}' for name in names_list
])
for i, name in enumerate(names_list):
    row, col = i // 3 + 1, i % 3 + 1
    h = results[name]['history']
    epochs = list(range(1, len(h['train_loss']) + 1))
    fig.add_trace(go.Scatter(x=epochs, y=h['train_loss'], name='Train',
                             showlegend=(i == 0)), row=row, col=col)
    fig.add_trace(go.Scatter(x=epochs, y=h['val_loss'], name='Validation',
                             showlegend=(i == 0)), row=row, col=col)
fig.update_layout(title='Сравнение архитектур: кривые обучения',
                  template='plotly_white', height=700, width=1200)
fig.show()


Что показывает: Accuracy и F1-score разных архитектур. Что даёт: помогает выбрать архитектуру с лучшим итоговым качеством.


In [ ]:
names = list(results.keys())
fig = go.Figure()
fig.add_trace(go.Bar(x=names, y=[results[n]['test_acc'] for n in names],
                     name='Accuracy', marker_color='#3498db'))
fig.add_trace(go.Bar(x=names, y=[results[n]['test_f1'] for n in names],
                     name='F1-score', marker_color='#e67e22'))
fig.update_layout(title='Сравнение архитектур MLP', yaxis_title='Метрика',
                  yaxis_range=[0.8, 1.0], barmode='group',
                  template='plotly_white', height=550, width=1000)
fig.show()


---
## 6. Извлечение градиентов при инференсе


In [ ]:
def extract_gradients(model, x_sample):
    """Извлекает градиенты для каждого линейного слоя при инференсе одного объекта."""
    model.eval()
    x = x_sample.clone().detach().unsqueeze(0)
    x.requires_grad_(True)

    activations = {}
    hooks = []

    def make_hook(name):
        def hook_fn(module, input, output):
            output.retain_grad()
            activations[name] = output
        return hook_fn

    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            h = module.register_forward_hook(make_hook(name))
            hooks.append(h)

    output = model(x)
    output.backward()

    grads = {}
    for name, act in activations.items():
        if act.grad is not None:
            grads[name] = act.grad.detach().numpy().flatten()

    for h in hooks:
        h.remove()

    return grads, activations


In [ ]:
sample_idx = 0
sample_x = X_test_t[sample_idx]
sample_label = y_test[sample_idx]
print(f'Выбран объект #{sample_idx}, истинный класс: {le.classes_[int(sample_label)]}')

grads, acts = extract_gradients(model, sample_x)

for layer_name, grad in grads.items():
    print(f'\nСлой: {layer_name}')
    print(f'  Размерность:       {grad.shape}')
    print(f'  Мин. градиент:     {grad.min():.6f}')
    print(f'  Макс. градиент:    {grad.max():.6f}')
    print(f'  Средний |градиент|: {np.abs(grad).mean():.6f}')


Выбран объект #0, истинный класс: B

Слой: network.0
  Размерность:       (64,)
  Мин. градиент:     -1.973763
  Макс. градиент:    2.674265
  Средний |градиент|: 0.445709

Слой: network.4
  Размерность:       (32,)
  Мин. градиент:     -0.844956
  Макс. градиент:    0.000000
  Средний |градиент|: 0.268751

Слой: network.8
  Размерность:       (1,)
  Мин. градиент:     1.000000
  Макс. градиент:    1.000000
  Средний |градиент|: 1.000000


Что показывает: градиенты отдельных нейронов для одного объекта. Что даёт: показывает, какие нейроны сильнее влияют на конкретное предсказание.


In [ ]:
n_layers = len(grads)
fig = make_subplots(rows=1, cols=n_layers,
                    subplot_titles=[f'Градиенты: {name}' for name in grads])
for i, (name, grad) in enumerate(grads.items()):
    fig.add_trace(go.Bar(x=list(range(len(grad))), y=grad,
                         marker_color='steelblue', showlegend=False), row=1, col=i + 1)
    fig.update_xaxes(title_text='Нейрон', row=1, col=i + 1)
    fig.update_yaxes(title_text='Градиент', row=1, col=i + 1)
fig.update_layout(title=f'Градиенты при инференсе объекта #{sample_idx}',
                  template='plotly_white', height=450, width=500 * n_layers)
fig.show()


Что показывает: средний абсолютный градиент каждого слоя. Что даёт: помогает сравнить влияние слоёв и заметить затухание градиента.


In [ ]:
layer_names = list(grads.keys())
mean_abs_grads = [np.abs(grads[name]).mean() for name in layer_names]
fig = go.Figure(go.Bar(x=layer_names, y=mean_abs_grads,
                       marker_color='#9b59b6', opacity=0.8))
fig.update_layout(title='Средний абсолютный градиент по слоям',
                  yaxis_title='Средний |градиент|', template='plotly_white',
                  height=450, width=750)
fig.show()


---
## 7. Функции активации

Реализация основных функций активации на NumPy и их визуализация.


In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def tanh(x):
    return np.tanh(x)

def relu(x):
    return np.maximum(0, x)

def leaky_relu(x, alpha=0.01):
    return np.where(x > 0, x, alpha * x)

def elu(x, alpha=1.0):
    return np.where(x > 0, x, alpha * (np.exp(x) - 1))

def swish(x):
    return x * sigmoid(x)

def softplus(x):
    return np.log(1 + np.exp(x))

def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))

Что показывает: форму восьми функций активации. Что даёт: позволяет сравнить, как функции преобразуют отрицательные и положительные значения.


In [ ]:
activations = {
    'Sigmoid': sigmoid, 'Tanh': tanh, 'ReLU': relu, 'Leaky ReLU': leaky_relu,
    'ELU': elu, 'Swish': swish, 'Softplus': softplus, 'GELU': gelu
}
x_range = np.linspace(-5, 5, 500)
colors = ['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3',
          '#a6d854', '#ffd92f', '#e5c494', '#b3b3b3']
fig = make_subplots(rows=2, cols=4, subplot_titles=list(activations.keys()))
for i, ((name, func), color) in enumerate(zip(activations.items(), colors)):
    row, col = i // 4 + 1, i % 4 + 1
    fig.add_trace(go.Scatter(x=x_range, y=func(x_range), mode='lines',
                             line=dict(color=color, width=2), showlegend=False), row=row, col=col)
    fig.add_hline(y=0, line_dash='dash', line_color='gray', line_width=0.5, row=row, col=col)
    fig.add_vline(x=0, line_dash='dash', line_color='gray', line_width=0.5, row=row, col=col)
fig.update_layout(title='Функции активации', template='plotly_white',
                  height=650, width=1200)
fig.show()


In [ ]:
print('Проверка значений функций активации при x = [-2, -1, 0, 1, 2]:')
test_x = np.array([-2, -1, 0, 1, 2])
for name, func in activations.items():
    vals = func(test_x)
    print(f'{name:12s}: {np.round(vals, 4)}')

Проверка значений функций активации при x = [-2, -1, 0, 1, 2]:
Sigmoid     : [0.1192 0.2689 0.5    0.7311 0.8808]
Tanh        : [-0.964  -0.7616  0.      0.7616  0.964 ]
ReLU        : [0 0 0 1 2]
Leaky ReLU  : [-0.02 -0.01  0.    1.    2.  ]
ELU         : [-0.8647 -0.6321  0.      1.      2.    ]
Swish       : [-0.2384 -0.2689  0.      0.7311  1.7616]
Softplus    : [0.1269 0.3133 0.6931 1.3133 2.1269]
GELU        : [-0.0454 -0.1588  0.      0.8412  1.9546]


---
## Выводы

1. **Датасет** содержит 569 наблюдений с 30 числовыми признаками, задача — бинарная классификация опухолей.
2. **Визуализация** показала, что классы разделимы по ряду признаков (radius, concavity, area), а некоторые признаки сильно коррелируют.
3. **Разбиение** 60/20/20 с сохранением пропорций классов (`stratify`) обеспечивает репрезентативность выборок.
4. **Базовый MLP [64, 32]** достигает высокого качества; кривые обучения позволяют выявить переобучение.
5. **Исследование архитектур** показало, что для данной задачи достаточно 1–2 скрытых слоёв; чрезмерная глубина не даёт преимущества.
6. **Градиенты** позволяют оценить вклад каждого слоя и нейрона в предсказание.
7. **Функции активации** реализованы на NumPy и визуализированы.
